# **Tasca S8.01. Analítica de dades en Pandas**

## Nivell 1 - EDA estructurat i neteja bàsica

### Exercici 1: Exploració inicial del dataset

Abans d'iniciar qualsevol procés analític sobre les dades d'empleats recollides per RRHH, cal fer una primera auditoria del dataset per confirmar-ne la mida, l'estructura i detectar possibles incidències de qualitat que condicionaran els següents passos de neteja.

**Algoritme:**
1. Carrego el fitxer Excel en un dataframe, amb un nom descriptiu perquè més endavant carregaré altres fitxers.
2. Faig un `head()` per comprovar visualment que la càrrega ha anat bé abans de fer res més.
3. Reviso quantes files i columnes té el dataset i quin tipus de dada té cada columna.
4. Miro una mostra de files a l'atzar (no només les primeres) per fer-me una idea real del contingut.
5. Compto files duplicades exactes i valors nuls per columna, sense corregir-los encara.
6. A partir d'aquests resultats, descric en paraules els problemes que detecto, per abordar-los als exercicis següents.

In [1]:
# importacions de totes les llibreries necessaries
import pandas as pd

import re
import numpy as np

import json

import unicodedata

import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
# carrego el dataset d'empleats; la fila 3 (comptant des de 0) conté els noms de columna
df_empleats = pd.read_excel(r'C:\Users\FDR\Documents\IT_Academy\Especialitat_en_Data_Analytics\sprint8\data\sprint8_complex.xlsx', header=3)

# li poso un nom descriptiu a la columna sense nom, ja que no és un índex tècnic sinó un identificador generat aleatòriament per seguretat (així ho indica el propi fitxer)
df_empleats = df_empleats.rename(columns={'Unnamed: 0': 'id_aleatori'})

# faig un head() abans de res per comprovar que el dataset s'ha carregat correctament
df_empleats.head()

,id_aleatori,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
0,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 â‚¬",True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1
1,1,Marc,MuÃ±oz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 â‚¬",NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7
2,2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,D,1358 euros,True,NaN,Grup A,TÃ¨cnic IT,4.0,NaN,NaN,NaN,16.7
3,3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,H,"1,478 â‚¬",True,NaN,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3
4,4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,"1,284 â‚¬",NaN,True,Grup B,Administratiu,NaN,True,NaN,11.8,13.1


In [3]:
# reviso les dimensions del dataset
print(f"Dimensions: {df_empleats.shape[0]} files x {df_empleats.shape[1]} columnes")

Dimensions: 1007 files x 20 columnes


In [4]:
# reviso les columnes i el tipus de dada de cadascuna
df_empleats.dtypes

id_aleatori                     int64
Nom                               str
Cognoms                           str
DNI                               str
País d'origen                     str
Ciutat                            str
Dia de Naixement              float64
Mes de Naixement              float64
Any de Naixement              float64
Gènere                            str
Salari mensual                    str
Fills                          object
No Fills                       object
Grup Professional                 str
CÃ rrec                           str
Nombre_fills                  float64
Te_cotxe                       object
Km_anuals                     float64
Consum_mitja_L_100km          float64
Temperatura_mitjana_ciutat    float64
dtype: object

In [5]:
# mostro una mostra representativa de files, no només les primeres
df_empleats.sample(10, random_state=42)

,id_aleatori,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
927,927,Anna,PÃ©rez Serra,81260169S,Espanya,Barcelona,11.0,2.0,1983.0,H,1.927 â‚¬,True,NaN,Grup B,Senior analyst,3.0,NaN,NaN,NaN,16.6
630,630,VÃ­ctor,DomÃ¨nech,22730042T,Espanya,Zaragoza,7.0,3.0,1995.0,H,"1,579 â‚¬",NaN,True,Grup B,TÃ¨cnic IT,0.0,NaN,NaN,NaN,11.8
682,682,Carmen,DÃ­az Casals,89650979N,Espanya,Murcia,11.0,10.0,1968.0,H,"2,425 â‚¬",NaN,True,Grup C,Analista junior,0.0,NaN,NaN,NaN,21.1
514,514,VÃ­ctor,Roca MuÃ±oz,25081699D,Espanya,Palma,8.0,2.0,1976.0,D,943â‚¬,True,NaN,Grup A,TÃ¨cnic IT,4.0,NaN,NaN,NaN,17.3
365,365,LucÃ­a,DomÃ¨nech,84449447A,Espanya,Zaragoza,1.0,6.0,1962.0,D,966 â‚¬,NaN,True,Grup A,Data Analyst,0.0,True,72237.0,10.8,15.4
655,655,VÃ­ctor,Serra,91229370H,Espanya,Palma,5.0,3.0,1968.0,D,"1,557 â‚¬",NaN,True,Grup B,Analista,NaN,True,29434.0,0.5,15.7
656,656,Felix,Richter,31742644W,Alemanya,Stuttgart,1.0,2.0,1966.0,H,tres mil,NaN,NaN,Grup A,Data Analyst,NaN,NaN,NaN,NaN,7.7
529,529,Alba,HernÃ¡ndez,43626780K,Espanya,Valladolid,16.0,8.0,1993.0,H,"1,664 â‚¬",True,NaN,Grup B,Analista junior,3.0,NaN,NaN,NaN,10.2
321,321,Ben,Koch,90293348E,Alemanya,Berlin,31.0,10.0,1980.0,F,"1,024 â‚¬",True,NaN,Grup A,Analista,3.0,NaN,NaN,NaN,9.0
70,70,Ã‰lise,Michel Garcia,57319709L,FranÃ§a,Nice,28.0,12.0,1960.0,D,1394â‚¬,True,NaN,Grup B,TÃ¨cnic IT,2.0,True,39561.0,NaN,16.5


In [6]:
# compto quantes files són completament duplicades
n_duplicats = df_empleats.duplicated().sum()
print(f"Files duplicades exactes: {n_duplicats}")

Files duplicades exactes: 0


In [7]:
# faig un primer repàs de valors nuls per columna
df_empleats.isnull().sum()

id_aleatori                     0
Nom                             1
Cognoms                         1
DNI                             1
País d'origen                   1
Ciutat                          1
Dia de Naixement                1
Mes de Naixement                1
Any de Naixement                1
Gènere                         41
Salari mensual                  1
Fills                         588
No Fills                      422
Grup Professional               2
CÃ rrec                         2
Nombre_fills                  298
Te_cotxe                      349
Km_anuals                     692
Consum_mitja_L_100km          746
Temperatura_mitjana_ciutat     71
dtype: int64

In [ ]:
# aplico un describe per veure l'estadística descriptiva de les variables numèriques
df_empleats.describe()

,id_aleatori,Dia de Naixement,Mes de Naixement,Any de Naixement,Nombre_fills,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
count,1007.000000,1006.000000,1006.000000,1006.000000,709.000000,315.000000,261.000000,936.000000
mean,503.000000,15.027833,8.556660,2002.229622,1.203103,55951.050794,11.024521,15.379167
std,290.840162,10.327436,14.298532,143.118077,1.389404,62095.437005,8.775557,7.276425
min,0.000000,-99.000000,0.000000,1940.000000,0.000000,-50.000000,0.500000,-10.000000
25%,251.500000,7.000000,3.250000,1969.000000,0.000000,22632.000000,4.700000,12.800000
50%,503.000000,15.000000,6.000000,1982.000000,1.000000,56135.000000,9.800000,15.500000
75%,754.500000,24.000000,10.000000,1995.000000,2.000000,80000.000000,13.600000,18.300000
max,1006.000000,32.000000,99.000000,3000.000000,5.000000,1000000.000000,25.000000,42.000000


In [ ]:
# aplico un info per veure el resum tècnic i l'estructura de les dades
df_empleats.info()

<class 'pandas.DataFrame'>
RangeIndex: 1007 entries, 0 to 1006
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id_aleatori                 1007 non-null   int64  
 1   Nom                         1006 non-null   str    
 2   Cognoms                     1006 non-null   str    
 3   DNI                         1006 non-null   str    
 4   País d'origen               1006 non-null   str    
 5   Ciutat                      1006 non-null   str    
 6   Dia de Naixement            1006 non-null   float64
 7   Mes de Naixement            1006 non-null   float64
 8   Any de Naixement            1006 non-null   float64
 9   Gènere                      966 non-null    str    
 10  Salari mensual              1006 non-null   str    
 11  Fills                       419 non-null    object 
 12  No Fills                    585 non-null    object 
 13  Grup Professional           1005 non-null   

**Coherència general i problemes detectats**

A primera vista, l'estructura del dataset és coherent (1007 registres, 20 columnes, sense files completament duplicades), però el contingut presenta diverses incidències que caldrà resoldre abans d'analitzar-lo:

- **Codificació de text incorrecta**: noms de columna (`CÃ rrec`) i valors de text (`PÃ©rez`, `MuÃ±oz`, `FranÃ§a`, `â‚¬`) mostren caràcters mal interpretats, típic d'un fitxer guardat originalment en UTF-8 i llegit amb una altra codificació.
- **`Salari mensual` sense format consistent**: conviuen com a mínim quatre formats diferents (`"1,469 â‚¬"`, `"1358 euros"`, `"943â‚¬"` sense espai, i fins i tot `"tres mil"` en lletres). Caldrà un procés de neteja i conversió a numèric abans de poder-hi treballar.
- **`Gènere` amb categories no normalitzades**: a la mostra ja apareixen `H`, `D`, `Dona`, `F`, cosa que fa pensar que un mateix concepte s'ha codificat de maneres diferents (i possiblement `F` correspon a "Femení" o a una altra llengua d'origen del registre, cosa que caldrà confirmar).
- **Redundància entre `Fills` i `No Fills`**: són dues columnes booleanes complementàries amb moltíssims nuls creuats (588 i 422 respectivament) — probablement cadascuna només s'omplia quan aplicava, i s'hauran d'unificar en una sola columna coherent.
- **Nuls concentrats en variables de mobilitat**: `Km_anuals` (692 nuls) i `Consum_mitja_L_100km` (746 nuls) tenen moltíssims valors buits, molt per sobre de la resta — caldrà comprovar si es correlacionen amb `Te_cotxe` (349 nuls també) abans de decidir com tractar-los.
- **Camps gairebé complets amb algun nul puntual**: `Nom`, `Cognoms`, `DNI`, `Ciutat`, `Salari mensual`, `Grup Professional` i `CÃ rrec` tenen només 1-2 nuls — són camps clau, per tant aquests pocs casos probablement acabaran sent files a descartar.
- **DNI encara no verificat**: no hem comprovat unicitat ni format del DNI — ho farem a l'Exercici 2.

Aquests problemes no es resolen en aquest exercici; només es descriuen com a punt de partida per a la neteja de l'Exercici 2 i la transformació de l'Exercici 3.

### Exercici 2: Validació i depuració del dataset

Abans de poder confiar en aquestes dades per a qualsevol anàlisi, cal garantir-ne la fiabilitat mínima: corregir problemes de format del text, confirmar que cada persona és identificable de manera única i decidir, de manera justificada, quins registres no es poden mantenir a l'anàlisi.

**Algoritme:**
1. Corregeixo la codificació incorrecta dels noms de columna i dels valors de text (mojibake per lectura UTF-8 interpretada com a Latin-1).
2. Torno a comprovar les files duplicades exactes ara que el text ja és net (per si la codificació incorrecta n'amagava alguna).
3. Comprovo si el DNI és únic per a cada persona.
4. Si hi ha DNIs duplicats, reviso la informació associada a cada parella/grup per decidir si són la mateixa persona amb dades inconsistents (i quin registre és més fiable) o un error clar a descartar.
5. Reviso els camps clau (Nom, Cognoms, DNI, Ciutat, Salari mensual, Grup Professional, Càrrec) i elimino només les files on falti informació irrecuperable en aquests camps, ja que no es poden imputar sense inventar-se dades.

In [10]:
# defineixo una funció per corregir el mojibake (text UTF-8 llegit incorrectament com a Latin-1)
def corregeix_codificacio(text):
    if isinstance(text, str):
        try:
            return text.encode('latin1').decode('utf-8')
        except (UnicodeEncodeError, UnicodeDecodeError):
            return text
    return text

In [11]:
# aplico la correcció als noms de columna
df_empleats.columns = [corregeix_codificacio(col) for col in df_empleats.columns]

In [12]:
# aplico la correcció a totes les columnes de tipus text
columnes_text = df_empleats.select_dtypes(include=['object', 'string']).columns
for col in columnes_text:
    df_empleats[col] = df_empleats[col].apply(corregeix_codificacio)

In [13]:
df_empleats.head()

,id_aleatori,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
0,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 â‚¬",True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1
1,1,Marc,Muñoz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 â‚¬",NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7
2,2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,D,1358 euros,True,NaN,Grup A,Tècnic IT,4.0,NaN,NaN,NaN,16.7
3,3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,H,"1,478 â‚¬",True,NaN,Grup B,Data Analyst,2.0,False,NaN,NaN,18.3
4,4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,"1,284 â‚¬",NaN,True,Grup B,Administratiu,NaN,True,NaN,11.8,13.1


In [14]:
# torno a comprovar duplicats exactes ara que el text ja està net
n_duplicats = df_empleats.duplicated().sum()
print(f"Files duplicades exactes: {n_duplicats}")

Files duplicades exactes: 0


In [15]:
# comprovo si el DNI identifica de manera única cada persona
n_dni_duplicats = df_empleats['DNI'].duplicated().sum()
print(f"DNIs duplicats: {n_dni_duplicats}")

DNIs duplicats: 15


In [16]:
# mostro totes les files implicades en un DNI duplicat, ordenades per DNI, per poder comparar-les
dnis_duplicats = df_empleats[df_empleats.duplicated(subset='DNI', keep=False)].sort_values('DNI')
dnis_duplicats

,id_aleatori,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
250,250,Laia,Navarro Vila,15909601K,Espanya,Bilbao,30.0,5.0,1958.0,D,1235 euros,True,True,Grup D,Analista junior,0.0,False,NaN,NaN,10.3
604,604,Laia,Navarro,15909601K,Espanya,Bilbao,30.0,5.0,1958.0,D,"1,676 â‚¬",True,False,Grup A,Analista,1.0,True,50.0,NaN,14.2
654,654,Oriol,Gómez,16618423V,Espanya,Madrid,10.0,1.0,1968.0,Home,1731 euros,True,NaN,Grup B,Data analyst,4.0,NaN,NaN,NaN,NaN
754,754,Oriol,Gómez,16618423V,Espanya,Barcelona,10.0,1.0,1968.0,Home,2.358 â‚¬,True,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN
758,758,Oriol,Gómez,16618423V,Espanya,Màlaga,10.0,1.0,1968.0,H,1879 euros,True,NaN,Grup B,Data Analyst,4.0,False,NaN,NaN,NaN
759,759,Oriol,Gómez,16618423V,Espanya,Sevilla,10.0,1.0,1968.0,NaN,"2,000 â‚¬",NaN,NaN,Grup A,Responsable de vendes,5.0,NaN,NaN,NaN,20.7
558,558,Alexia,Pérez Díaz,49529438G,Espanya,Màlaga,12.0,10.0,1981.0,Dona,"1,176 â‚¬",NaN,True,Grup A,Data Analyst,0.0,False,NaN,NaN,38.0
692,692,Alexia,Pérez Díaz,49529438G,Espanya,Màlaga,12.0,10.0,1981.0,Dona,1663â‚¬,NaN,True,Grup C,Cap de projecte,NaN,True,80000.0,10.7,16.9
757,757,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786â‚¬,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9
1004,1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15.0,7.0,1973.0,D,786â‚¬,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9


In [17]:
# defineixo els camps clau que no haurien de contenir nuls
camps_clau = ['Nom', 'Cognoms', 'DNI', 'Ciutat', 'Salari mensual', 'Grup Professional', 'Càrrec']

# mostro les files que tenen algun nul en aquests camps
files_incompletes = df_empleats[df_empleats[camps_clau].isnull().any(axis=1)]
files_incompletes

,id_aleatori,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
754,754,Oriol,Gómez,16618423V,Espanya,Barcelona,10.0,1.0,1968.0,Home,2.358 â‚¬,True,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN
1005,1005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# defineixo les columnes que identifiquen si dues files són "la mateixa persona"
columnes_identitat = ['Nom', 'Cognoms', 'Any de Naixement']

files_a_eliminar = []

# recorro cada grup de DNI duplicat
for dni, grup in df_empleats[df_empleats.duplicated(subset='DNI', keep=False)].groupby('DNI'):

    identitats_uniques = grup[columnes_identitat].drop_duplicates()

    if len(identitats_uniques) > 1:
        # dins del mateix DNI hi ha persones amb noms/cognoms/naixement diferents:
        # el DNI no és fiable per identificar-les, així que elimino tot el grup
        files_a_eliminar.extend(grup.index.tolist())
    else:
        # és la mateixa persona repetida: em quedo amb la fila més completa (menys nuls)
        grup_ordenat = grup.assign(n_nuls=grup.isnull().sum(axis=1)).sort_values('n_nuls')
        files_a_mantenir = grup_ordenat.index[0]
        files_a_eliminar.extend([i for i in grup.index if i != files_a_mantenir])

print(f"Files a eliminar per DNI no fiable o duplicat: {len(files_a_eliminar)}")

Files a eliminar per DNI no fiable o duplicat: 18


In [19]:
df_empleats = df_empleats.drop(index=files_a_eliminar)
print(f"Dimensions després de resoldre els DNI duplicats: {df_empleats.shape}")

Dimensions després de resoldre els DNI duplicats: (989, 20)


In [20]:
# la fila amb id_aleatori 1005 no té cap valor a cap columna: és irrecuperable, l'elimino
df_empleats = df_empleats.dropna(how='all', subset=[c for c in df_empleats.columns if c != 'id_aleatori'])
print(f"Dimensions finals: {df_empleats.shape}")

Dimensions finals: (988, 20)


In [21]:
# torno a comprovar si queda alguna fila amb valors nuls en els camps clau, ara que ja he resolt els DNIs duplicats i la fila completament buida
camps_clau = ['Nom', 'Cognoms', 'DNI', 'Ciutat', 'Salari mensual', 'Grup Professional', 'Càrrec']
files_incompletes = df_empleats[df_empleats[camps_clau].isnull().any(axis=1)]
files_incompletes

,id_aleatori,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Càrrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat


In [22]:
# comprovo les dimensions finals del dataset després de tota la depuració de l'exercici 2
print(f"Dimensions finals: {df_empleats.shape}")

Dimensions finals: (988, 20)


**Decisions preses sobre duplicats i DNI**

- No hi ha files completament duplicades (Exercici 1) perquè `id_aleatori` és diferent en cada registre, encara que la resta de camps coincideixin; per això calia una comprovació addicional específica de duplicats reals un cop corregida la codificació.
- 15 files estaven implicades en DNIs duplicats, repartides en 9 grups de DNI:
  - En els grups on el nom, els cognoms i l'any de naixement coincidien (p. ex. Oriol Ferrer, Ida Karlsen Eriksen, Claudia Gil Ferrer, Alba Muñoz), es tracta de la mateixa persona introduïda dues vegades: s'ha mantingut un únic registre.
  - En els grups on el DNI apareixia associat a persones amb noms, cognoms o any de naixement clarament diferents (p. ex. Guillem Pujol Costa / Marta Ruiz, o el grup de 4 identitats diferents sota el DNI 90585882A), el DNI no es pot considerar fiable com a identificador únic; s'han descartat totes les files d'aquests grups, ja que no hi ha manera de determinar quina persona és la titular real.
  - En els grups amb la mateixa identitat però dades analítiques inconsistents (Oriol Gómez, Alexia Pérez Díaz, Laia Navarro), s'ha conservat el registre amb menys valors nuls, per ser el més complet i, per tant, el més fiable.
- S'ha eliminat una fila completament buida (`id_aleatori` 1005), sense cap altre valor a cap columna, ja que no aporta cap informació recuperable.
- Un cop resoltes aquestes dues qüestions, cap fila restant té valors nuls en els camps clau (Nom, Cognoms, DNI, Ciutat, Salari mensual, Grup Professional, Càrrec), de manera que no ha calgut cap eliminació addicional per aquest motiu. Dimensions finals: 988 files x 20 columnes.

### Exercici 3: Transformació i preparació del dataset

Abans de poder analitzar el dataset amb garanties, cal assegurar que les categories siguin consistents, els camps numèrics estiguin ben tipats i els valors mancants es tractin amb un criteri justificat. Aquest pas prepara les dades perquè les anàlisis posteriors reflecteixin la realitat i no artefactes de com s'han recollit.

**Algoritme:**
1. Reviso els valors únics de `Gènere` (la principal columna categòrica amb inconsistències detectades) i els normalitzo a un únic format.
2. Netejo i converteixo `Salari mensual` a un tipus numèric, ja que ara és text amb formats diferents (comes, punts, "euros", paraules).
3. Comprovo la coherència entre `Te_cotxe`, `Km_anuals` i `Consum_mitja_L_100km`: si algú té dades de consum o quilometratge, ha de constar que té cotxe; decideixo com tractar els casos amb consum però sense quilometratge.
4. Unifico `Fills`, `No Fills` i `Nombre_fills` en una única columna coherent.
5. Valido que la temperatura sigui la mateixa per a totes les persones d'una mateixa ciutat, i imputo els casos on falti, primer a nivell de ciutat i, si cal, a nivell de país.
6. Imputo els salaris que falten a partir del Grup Professional i el Càrrec, ja que són els factors amb més relació directa amb el salari.

In [23]:
# reviso totes les categories diferents que hi ha a Gènere, incloent-hi els nuls, abans de decidir com normalitzar-les
df_empleats['Gènere'].value_counts(dropna=False)

Gènere
D       317
H       275
Dona    118
F       107
Home     50
NaN      39
M        34
A        24
NC       23
??        1
Name: count, dtype: int64

In [24]:
# defineixo com es tradueix cada variant que hem vist a un únic valor estàndard
mapatge_genere = {
    'h': 'Home', 'home': 'Home', 'm': 'Home',
    'd': 'Dona', 'dona': 'Dona', 'f': 'Dona',
    'a': 'Altre',
    'nc': np.nan, '??': np.nan
}

# passo el text a minúscules i sense espais abans de mapejar-lo
df_empleats['Gènere'] = df_empleats['Gènere'].str.lower().str.strip().map(mapatge_genere)

In [25]:
# verifico que els canvis s'han realitzat correctament
df_empleats['Gènere'].value_counts(dropna=False)

Gènere
Dona     542
Home     359
NaN       63
Altre     24
Name: count, dtype: int64

In [26]:
# guardo el text original del salari abans de convertir-lo, per poder revisar després els casos que fallin
salari_original = df_empleats['Salari mensual'].copy()

paraules_a_numeros = {
    'un': 1, 'una': 1, 'dos': 2, 'dues': 2, 'tres': 3, 'quatre': 4, 'cinc': 5,
    'sis': 6, 'set': 7, 'vuit': 8, 'nou': 9, 'deu': 10, 'mil': 1000
}

def converteix_salari(valor):
    if pd.isnull(valor):
        return np.nan
    text = str(valor).lower()
    text = text.replace('â‚¬', '').replace('€', '').replace('euros', '').replace('eur', '').strip()
    if re.fullmatch(r'[\d.,]+', text):
        numero_net = text.replace('.', '').replace(',', '')
        return int(numero_net)
    total = 0
    for paraula in text.split():
        if paraula in paraules_a_numeros:
            valor_paraula = paraules_a_numeros[paraula]
            total = total * 1000 if valor_paraula == 1000 and total > 0 else (1000 if valor_paraula == 1000 else total + valor_paraula)
    return total if total > 0 else np.nan

df_empleats['Salari mensual'] = df_empleats['Salari mensual'].apply(converteix_salari)

In [27]:
# comprovo el tipus de dada resultant i si hi ha algun valor que no s'hagi pogut convertir
print(df_empleats['Salari mensual'].dtype)
print(f"Salaris nuls després de la conversió: {df_empleats['Salari mensual'].isnull().sum()}")
df_empleats['Salari mensual'].describe()

float64
Salaris nuls després de la conversió: 15


count     973.000000
mean     1559.173690
std       625.550689
min       673.000000
25%      1075.000000
50%      1400.000000
75%      1895.000000
max      3672.000000
Name: Salari mensual, dtype: float64

In [28]:
# reviso el text original dels salaris que no s'han pogut convertir
indexs_afectats = df_empleats[df_empleats['Salari mensual'].isnull()].index
salari_original.loc[indexs_afectats]

93        n/d
146       n/d
159       n/d
254       n/d
291       n/d
307       n/d
384       n/d
446       n/d
860       n/d
875       n/d
917       n/d
953       n/d
955       n/d
956       n/d
1006    molts
Name: Salari mensual, dtype: str

In [29]:
# imputo amb la mediana del mateix Grup Professional + Càrrec
df_empleats['Salari mensual'] = df_empleats.groupby(['Grup Professional', 'Càrrec'])['Salari mensual'].transform(lambda x: x.fillna(x.median()))

# si encara en queda algun, ho torno a provar només per Grup Professional
df_empleats['Salari mensual'] = df_empleats.groupby('Grup Professional')['Salari mensual'].transform(lambda x: x.fillna(x.median()))

print(f"Salaris nuls després d'imputar: {df_empleats['Salari mensual'].isnull().sum()}")

Salaris nuls després d'imputar: 1


In [30]:
# identifico quina fila queda amb salari nul i de quin grup professional/càrrec és
df_empleats[df_empleats['Salari mensual'].isnull()][['Grup Professional', 'Càrrec']]

,Grup Professional,Càrrec
1006,Z,???


In [31]:
# casos on hi ha dades de quilometratge o consum, però Te_cotxe no diu True
incoherencia_cotxe = df_empleats[
    (df_empleats['Km_anuals'].notna() | df_empleats['Consum_mitja_L_100km'].notna())
    & (df_empleats['Te_cotxe'] != True)
]
print(f"Files amb dades de cotxe però Te_cotxe no és True: {len(incoherencia_cotxe)}")

# casos on hi ha consum mitjà però no hi ha quilometratge anual
incoherencia_consum_sense_km = df_empleats[
    df_empleats['Consum_mitja_L_100km'].notna() & df_empleats['Km_anuals'].isna()
]
print(f"Files amb consum però sense quilometratge: {len(incoherencia_consum_sense_km)}")

Files amb dades de cotxe però Te_cotxe no és True: 4
Files amb consum però sense quilometratge: 2


In [32]:
# si hi ha dades de km o consum, és evident que la persona té cotxe: corregeixo Te_cotxe
condicio = (df_empleats['Km_anuals'].notna() | df_empleats['Consum_mitja_L_100km'].notna())
df_empleats.loc[condicio, 'Te_cotxe'] = True

# si té consum mitjà però no quilometratge, no puc inventar-me quants km fa: deixo Km_anuals com a valor desconegut, ja que estimar-lo sense cap altra informació introduiria un biaix no justificat

In [33]:
# comprovo si hi ha contradiccions: algú marcat com "No Fills" però amb Nombre_fills > 0
contradiccio = df_empleats[(df_empleats['No Fills'] == True) & (df_empleats['Nombre_fills'] > 0)]
print(f"Contradiccions No Fills / Nombre_fills: {len(contradiccio)}")

# construeixo una única columna: prioritzo la informació explícita (Fills/No Fills) per sobre del comptador
def unifica_fills(fila):
    if fila['No Fills'] == True:
        return 0
    if fila['Fills'] == True and pd.notna(fila['Nombre_fills']):
        return fila['Nombre_fills']
    if fila['Fills'] == True:
        # sabem que en té, però no quants: no ho puc deixar en blanc perquè seria "sense fills"
        return np.nan
    return fila['Nombre_fills']  # si no hi ha cap indicació booleana, em quedo amb el que digui el comptador

df_empleats['Nombre_fills'] = df_empleats.apply(unifica_fills, axis=1)

# ara ja puc eliminar les columnes redundants
df_empleats = df_empleats.drop(columns=['Fills', 'No Fills'])

Contradiccions No Fills / Nombre_fills: 20


In [34]:
# verifico que els canvis s'hagin realitzat correctament
print(df_empleats['Nombre_fills'].isnull().sum())
df_empleats[['Nombre_fills']].describe()

30


,Nombre_fills
count,958.000000
mean,0.824635
std,1.271231
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,4.000000


In [35]:
# comprovo si una mateixa ciutat té més d'un valor diferent de temperatura
temp_per_ciutat = df_empleats.groupby('Ciutat')['Temperatura_mitjana_ciutat'].nunique()
ciutats_incoherents = temp_per_ciutat[temp_per_ciutat > 1]
print(f"Ciutats amb valors de temperatura inconsistents: {len(ciutats_incoherents)}")

# miro els valors reals d'unes quantes ciutats per veure si la diferència és petita o real
df_empleats[df_empleats['Ciutat'].isin(ciutats_incoherents.index[:8])][['Ciutat', 'Temperatura_mitjana_ciutat']].sort_values('Ciutat')

Ciutats amb valors de temperatura inconsistents: 44


,Ciutat,Temperatura_mitjana_ciutat
32,A Coruña,15.6
50,A Coruña,16.2
54,A Coruña,15.5
106,A Coruña,15.9
110,A Coruña,10.0
...,...,...
666,Bordeaux,-5.0
706,Bordeaux,15.2
820,Bordeaux,NaN
860,Bordeaux,16.2


In [36]:
# imputo la temperatura que falta amb la mediana de la mateixa ciutat (robusta davant de valors extrems)
df_empleats['Temperatura_mitjana_ciutat'] = df_empleats.groupby('Ciutat')['Temperatura_mitjana_ciutat'].transform(lambda x: x.fillna(x.median()))

# si encara queda alguna ciutat sense cap valor conegut, imputo amb la mediana del país d'origen
df_empleats['Temperatura_mitjana_ciutat'] = df_empleats.groupby("País d'origen")['Temperatura_mitjana_ciutat'].transform(lambda x: x.fillna(x.median()))

print(f"Temperatures encara nul·les: {df_empleats['Temperatura_mitjana_ciutat'].isnull().sum()}")

Temperatures encara nul·les: 1


In [37]:
# identifico quina fila queda amb temperatura nul·la i de quina ciutat/país és
df_empleats[df_empleats['Temperatura_mitjana_ciutat'].isnull()][['Ciutat', "País d'origen"]]

,Ciutat,País d'origen
1006,Desconeguda,Narnia


In [38]:
# elimino la fila 1006: és una fila de prova/no vàlida (Grup "Z", Càrrec "???", Ciutat "Desconeguda", País "Narnia"), no dades reals incompletes, per tant no té sentit imputar-la
df_empleats = df_empleats.drop(index=1006)
print(f"Dimensions després d'eliminar la fila no vàlida: {df_empleats.shape}")

Dimensions després d'eliminar la fila no vàlida: (987, 18)


In [39]:
print(f"Salaris nuls: {df_empleats['Salari mensual'].isnull().sum()}")
print(f"Temperatures nul·les: {df_empleats['Temperatura_mitjana_ciutat'].isnull().sum()}")

Salaris nuls: 0
Temperatures nul·les: 0


**Resum de les transformacions aplicades**

- **Gènere**: normalitzat a 4 categories úniques (Home, Dona, Altre, i nul per als casos "No consta"/"??"), unificant les variants de majúscules, abreviatures i idiomes detectades.
- **Salari mensual**: convertit de text a numèric, gestionant formats amb separadors de milers, el símbol €, la paraula "euros" i imports escrits amb lletres. Els 15 casos no convertibles ("n/d" i "molts") s'han imputat amb la mediana del seu Grup Professional i Càrrec.
- **Cotxe i mobilitat**: corregit `Te_cotxe` a `True` en els 4 casos amb dades de quilometratge o consum enregistrades. Els 2 casos amb consum però sense quilometratge s'han deixat sense imputar, ja que no hi ha cap variable fiable per estimar-lo sense introduir un biaix arbitrari.
- **Fills**: unificades les columnes `Fills`, `No Fills` i `Nombre_fills` en una única columna coherent, la de `Nombre_fills`; s'han eliminat les dues columnes redundants. Donem per entés que els casos en el quals hi hagi especificat un determinat nombre de fills, significa que tenen fills.
- **Temperatura mitjana**: imputada amb la mediana de la mateixa ciutat (robusta davant valors extrems, com el -5.0 detectat a Bordeaux) i, en darrer terme, amb la mediana del país d'origen.
- **Fila no vàlida**: eliminat el registre 1006 (Grup Professional "Z", Càrrec "???", Ciutat "Desconeguda", País "Narnia", Salari "molts"), per tractar-se d'una fila de prova sense cap dada real ni imputable, no d'un cas d'informació incompleta.
- Dimensions finals després de la transformació: 987 files x 18 columnes.

Queden per analitzar les dates de naixement, ja que moltes d'aquestes son incongruents tant pel que fa a alguns dies, mesos i anys. Però no ha sigut possible per falta de temps. Ho continuaré més endavant.

###  Exercici 4: Primeres anàlisis descriptives

Amb el dataset ja net i transformat, és moment de fer una primera lectura de conjunt: quines diferències salarials es detecten entre col·lectius, i com varien aquestes diferències en creuar-les amb el país d'origen.

**Algoritme:**
1. Construeixo una taula resum amb el sou mig, medià, mínim i màxim per Gènere, ordenada pel sou mig.
2. Construeixo una taula creuada amb el salari mig per Gènere (files) i País d'origen (columnes), amb les mitjanes als marges i format condicional per ressaltar els valors més alts.

In [40]:
# agrupo per gènere i calculo mitjana, mediana, mínim i màxim del salari
resum_salari_genere = df_empleats.groupby('Gènere')['Salari mensual'].agg(['mean', 'median', 'min', 'max'])

# ordeno la taula pel sou mig, de més alt a més baix
resum_salari_genere = resum_salari_genere.sort_values('mean', ascending=False)

resum_salari_genere

,mean,median,min,max
Gènere,,,,
Altre,1690.375000,1205.5,673.0,3600.0
Home,1690.250696,1501.0,726.0,3672.0
Dona,1473.085793,1333.5,701.0,3564.0


In [41]:
# taula creuada: salari mig per gènere (files) i país (columnes), amb marges (totals) i format condicional
taula_creuada = pd.crosstab(
    index=df_empleats['Gènere'],
    columns=df_empleats["País d'origen"],
    values=df_empleats['Salari mensual'],
    aggfunc='mean',
    margins=True,
    margins_name='Mitjana'
)

taula_creuada.style.background_gradient(cmap='Oranges', axis=None)

País d'origen,Alemanya,Espanya,França,Itàlia,Noruega,Mitjana
Gènere,,,,,,
Altre,nan,1774.095238,1104.333333,nan,nan,1690.375000
Dona,1429.909091,1469.349206,1403.595745,1452.141667,1779.750000,1473.085793
Home,2103.818182,1701.187023,1658.645161,1640.729730,1434.555556,1690.250696
Mitjana,1598.386364,1570.924357,1490.123457,1524.077320,1631.809524,1563.007027


In [42]:
# reviso quantes persones hi ha a cada grup, ja que sense saber-ho no es pot valorar si les mitjanes són comparables
mida_grups = df_empleats['Gènere'].value_counts()
mida_grups

Gènere
Dona     542
Home     359
Altre     24
Name: count, dtype: int64

In [43]:
# calculo el salari mig de cada gènere com a percentatge respecte al salari mig global, per veure la diferència relativa en lloc de l'import absolut en euros
salari_mig_global = df_empleats['Salari mensual'].mean()

resum_relatiu = resum_salari_genere.copy()
resum_relatiu['n'] = mida_grups
resum_relatiu['ratio_vs_global'] = (resum_relatiu['mean'] / salari_mig_global * 100).round(1)

resum_relatiu

,mean,median,min,max,n,ratio_vs_global
Gènere,,,,,,
Altre,1690.375000,1205.5,673.0,3600.0,24,108.5
Home,1690.250696,1501.0,726.0,3672.0,359,108.5
Dona,1473.085793,1333.5,701.0,3564.0,542,94.6


**Interpretació dels resultats**

- Els homes tenen el salari mig més alt (1690€), seguits molt de prop per la categoria "Altre" (1690€), i les dones queden per sota (1473€) — una bretxa d'uns 217€ mensuals entre homes i dones.
- La taula creuada per país mostra que aquesta bretxa no és uniforme: a Alemanya és molt marcada (2104€ homes vs. 1430€ dones), mentre que a Noruega el patró s'inverteix (1435€ homes vs. 1780€ dones).
- Els `NaN` a la taula creuada corresponen a combinacions gènere-país sense cap persona representada en el dataset (per exemple, cap persona "Altre" registrada d'Alemanya, Itàlia o Noruega), no a un error de càlcul.

*Per què cal mirar també percentatges i mides de mostra:*

- En valors absoluts, "Altre" i "Home" semblen pràcticament idèntics (1690€ de mitjana als dos casos), i el percentatge respecte a la mitjana global ho confirma amb una coincidència gairebé exacta (108.5% als dos grups). Però "Altre" té només 24 persones davant les 359 d'"Home": la mateixa xifra final amaga un nivell de fiabilitat molt diferent, ja que amb una mostra de 24 persones un parell de salaris alts o baixos concrets poden desplaçar la mitjana amb molta més facilitat.
- Dona (542 persones, el grup més gran) queda clarament per sota de la mitjana global (94.6%), una diferència que sí que es pot considerar robusta per la mida de la mostra.
- A la taula creuada per país el problema s'agreuja: algunes combinacions gènere-país tenen molt poques persones, de manera que comparar-ne el salari mig amb el d'una combinació amb desenes de registres, com si totes dues xifres tinguessin el mateix pes, portaria a conclusions poc fiables si no es té en compte la mida de cada cel·la.

## Nivell 2 - Anàlisi amb dades enriquides i estructures múltiples

### Exercici 1: Enriquiment amb dades externes

Per calcular un increment salarial ajustat i realista, cal incorporar informació que no ve del propi departament de RRHH: la política salarial vigent de l'empresa i el context econòmic de cada ciutat on viuen els empleats. Abans d'aplicar cap increment, cal garantir que aquestes fonts externes es puguin combinar correctament amb el dataset ja net.


**Algoritme:**
1. Carrego el fitxer CSV de context de ciutats i el fitxer JSON de política salarial, i n'inspecciono l'estructura abans de fer-hi res.
2. Uneixo el dataset d'empleats amb el context de ciutats, comprovant que no quedin registres sense aparellar.
3. A partir de la política salarial, calculo l'increment corresponent a cada empleat combinant Grup Professional, situació familiar, franja d'edat i cost de vida de la ciutat.
4. Identifico els registres que no poden rebre algun d'aquests ajustos i decideixo, de manera justificada, com tractar-los en el càlcul final.

In [44]:
# carrego el fitxer amb informació contextual de ciutats europees
# especifico encoding='latin1' perquè el fitxer no està guardat en UTF-8
df_ciutats = pd.read_csv(r'C:\Users\FDR\Documents\IT_Academy\Especialitat_en_Data_Analytics\sprint8\data\ciutats_context.csv', encoding='latin1')
df_ciutats.head()

,Ciutat,País,Cost_vida_index,Transport_public_score,Clima_tipus
0,Barcelona,Espanya,78,8.5,Mediterrani
1,Madrid,Espanya,75,8.0,Continental
2,València,Espanya,70,7.5,Mediterrani
3,Sevilla,Espanya,65,7.0,Mediterrani
4,Bilbao,Espanya,72,7.8,Oceànic


In [45]:
# carrego el fitxer JSON amb les regles d'increment salarial
with open(r'C:\Users\FDR\Documents\IT_Academy\Especialitat_en_Data_Analytics\sprint8\data\politica_salarial.json', 'r', encoding='utf-8') as f:
    politica_salarial = json.load(f)

politica_salarial

{'base_increment_by_group': {'Grup A': 5.0,
  'Grup B': 3.5,
  'Grup C': 2.0,
  'Grup D': 8.0},
 'children_adjustment': {'has_children': 1.5, 'no_children': 0.0},
 'age_adjustment': {'18-29': 0.0, '30-44': 1.0, '45-59': 2.0, '60+': 1.5},
 'cost_of_living_adjustment': {'low': 0.0, 'medium': 1.0, 'high': 2.0},
 'transport_adjustment': {'low': 0.5, 'medium': 0.0, 'high': -0.5},
 'rules': {'max_total_increment_percent': 15, 'min_salary_threshold': 800}}

In [46]:
# miro totes les ciutats que sí que té el fitxer de context, per comparar-les amb les que falten
sorted(df_ciutats['Ciutat'].unique())

['A Coru\\u00f1a',
 'Alacant',
 'Barcelona',
 'Bergen',
 'Berlin',
 'Bilbao',
 'Bordeaux',
 'Firenze',
 'Frankfurt',
 'Genova',
 'Girona',
 'Hamburg',
 'Las Palmas de Gran Canaria',
 'Leipzig',
 'Lille',
 'Lyon',
 'Madrid',
 'Marseille',
 'Milano',
 'Munchen',
 'Murcia',
 'Napoli',
 'Oslo',
 'Palma',
 'Paris',
 'Roma',
 'Sevilla',
 'Stuttgart',
 'Tarragona',
 'Torino',
 'Valencia',
 'Valladolid',
 'València',
 'Venezia',
 'Zaragoza']

In [47]:
def normalitza_ciutat(nom):
    # elimino accents i converteixo a minúscules per poder comparar noms escrits de manera diferent
    if pd.isnull(nom):
        return nom
    nom_sense_accents = unicodedata.normalize('NFKD', nom).encode('ascii', 'ignore').decode('utf-8')
    return nom_sense_accents.lower().strip()

# corregeixo el cas concret d'A Coruña: el fitxer conté "\u00f1" com a text literal en lloc del caràcter ñ
df_ciutats['Ciutat'] = df_ciutats['Ciutat'].str.replace(r'\\u00f1', 'ñ', regex=True)

# normalitzo ciutat i país a totes dues taules
df_empleats['_ciutat_normalitzada'] = df_empleats['Ciutat'].apply(normalitza_ciutat)
df_empleats['_pais_normalitzat'] = df_empleats["País d'origen"].apply(normalitza_ciutat)
df_ciutats['_ciutat_normalitzada'] = df_ciutats['Ciutat'].apply(normalitza_ciutat)
df_ciutats['_pais_normalitzat'] = df_ciutats['País'].apply(normalitza_ciutat)

# uneixo per la combinació de ciutat i país normalitzats, per no confondre ciutats homònimes de països diferents
df_empleats = df_empleats.merge(
    df_ciutats.drop(columns=['Ciutat', 'País']),
    on=['_ciutat_normalitzada', '_pais_normalitzat'],
    how='left'
)

print(f"Dimensions després de la unió: {df_empleats.shape}")

sense_context = sorted(df_empleats[df_empleats['Cost_vida_index'].isnull()]['Ciutat'].unique())
n_afectats = df_empleats['Cost_vida_index'].isnull().sum()
print(f"Persones afectades per manca de context: {n_afectats} de {len(df_empleats)}")
print(f"Ciutats sense context: {sense_context}")

Dimensions després de la unió: (987, 23)
Persones afectades per manca de context: 132 de 987
Ciutats sense context: ['Bologna', 'Düsseldorf', 'Köln', 'Màlaga', 'Nantes', 'Nice', 'Stavanger', 'Toulouse', 'Tromsø', 'Trondheim']


In [48]:
# calculo la franja d'edat de cada persona a partir de l'any de naixement
any_actual = 2026
df_empleats['edat'] = any_actual - df_empleats['Any de Naixement']

def franja_edat(edat):
    if pd.isnull(edat):
        return np.nan
    if edat < 30:
        return '18-29'
    if edat < 45:
        return '30-44'
    if edat < 60:
        return '45-59'
    return '60+'

df_empleats['franja_edat'] = df_empleats['edat'].apply(franja_edat)

# calculo la categoria de cost de vida a partir de l'índex numèric, amb llindars raonables
def categoria_cost_vida(index):
    if pd.isnull(index):
        return np.nan
    if index < 65:
        return 'low'
    if index < 80:
        return 'medium'
    return 'high'

df_empleats['categoria_cost_vida'] = df_empleats['Cost_vida_index'].apply(categoria_cost_vida)

In [49]:
# calculo l'increment total sumant els quatre components de la política salarial
def calcula_increment(fila):
    increment = politica_salarial['base_increment_by_group'].get(fila['Grup Professional'], 0)

    te_fills = fila['Nombre_fills'] > 0 if pd.notna(fila['Nombre_fills']) else False
    increment += politica_salarial['children_adjustment']['has_children' if te_fills else 'no_children']

    if pd.notna(fila['franja_edat']):
        increment += politica_salarial['age_adjustment'][fila['franja_edat']]

    if pd.notna(fila['categoria_cost_vida']):
        increment += politica_salarial['cost_of_living_adjustment'][fila['categoria_cost_vida']]

    increment_maxim = politica_salarial['rules']['max_total_increment_percent']
    return min(increment, increment_maxim)

df_empleats['increment_percent'] = df_empleats.apply(calcula_increment, axis=1)
df_empleats['salari_ajustat'] = df_empleats['Salari mensual'] * (1 + df_empleats['increment_percent'] / 100)

df_empleats[['Salari mensual', 'increment_percent', 'salari_ajustat']].describe()

,Salari mensual,increment_percent,salari_ajustat
count,987.000000,987.000000,987.000000
mean,1557.811044,6.672746,1660.176388
std,625.465957,1.711226,668.948644
min,673.000000,2.000000,713.380000
25%,1073.500000,5.500000,1141.665000
50%,1397.000000,6.500000,1486.495000
75%,1895.000000,8.000000,2006.082500
max,3672.000000,12.500000,4094.280000


In [50]:
# identifico i compto els registres que no han pogut rebre l'ajust de cost de vida per manca de context de ciutat
sense_ajust_cost_vida = df_empleats[df_empleats['categoria_cost_vida'].isnull()]
print(f"Persones sense ajust de cost de vida aplicat: {len(sense_ajust_cost_vida)}")

Persones sense ajust de cost de vida aplicat: 132


**Decisions preses en l'enriquiment amb dades externes**

- La unió amb el context de ciutats s'ha fet per la combinació de ciutat i país, no només per ciutat, ja que el fitxer conté ciutats homònimes de països diferents (Paris a França i als Estats Units; València a Espanya i Valencia a Veneçuela) que una unió només pel nom hauria confós, duplicant registres.
- S'ha corregit un cas de codificació al fitxer de ciutats, on "A Coruña" contenia el caràcter especial escrit literalment com a text (`\u00f1`) en lloc del símbol real.
- 132 persones (de 987) pertanyen a 10 ciutats no cobertes pel fitxer de context (Bologna, Düsseldorf, Köln, Màlaga, Nantes, Nice, Stavanger, Toulouse, Tromsø i Trondheim). No se'ls ha imputat cap valor de cost de vida artificial: senzillament no reben aquest component de l'increment, en comptes de forçar una estimació sense base real.
- L'increment total s'ha calculat combinant Grup Professional, situació familiar (té fills o no), franja d'edat i cost de vida de la ciutat, aplicant el límit màxim del 15% que marca la política. L'increment resultant oscil·la entre el 2% i el 12.5%, amb una mitjana del 6.67%, i cap cas ha arribat a topar amb el límit.
- El salari ajustat s'ha calculat aplicant aquest percentatge d'increment sobre el salari mensual net ja transformat a l'Exercici 3.

### Exercici 2: Reestructuració del dataset

Per poder comparar diferents tipus de mesures (salaris, mobilitat, context climàtic) d'una manera homogènia, cal transformar el dataset del format ample actual —amb una columna per a cada variable— a un format llarg, on cada fila representi una única mesura d'una persona. Aquest format facilita les agregacions i comparacions que farem als propers exercicis.

**Algoritme:**
1. Defineixo quines columnes actuen com a identificadors de la persona (no canvien de format) i quines representen valors mesurats (les que passaran a format llarg).
2. Aplico `melt()` amb aquesta distinció.
3. Comprovo que el resultat té la mida esperada i l'estructura correcta.

In [51]:
# defineixo les columnes que identifiquen la persona i el seu context (no es reestructuren)
columnes_id = [
    'id_aleatori', 'Nom', 'Cognoms', 'DNI', "País d'origen", 'Ciutat',
    'Gènere', 'Grup Professional', 'Càrrec', 'franja_edat', 'categoria_cost_vida'
]

# defineixo les columnes que representen valors mesurats (passen a format llarg)
columnes_valor = [
    'Salari mensual', 'salari_ajustat', 'increment_percent',
    'Km_anuals', 'Consum_mitja_L_100km',
    'Temperatura_mitjana_ciutat', 'Cost_vida_index', 'Transport_public_score'
]

df_llarg = df_empleats.melt(
    id_vars=columnes_id,
    value_vars=columnes_valor,
    var_name='variable',
    value_name='valor'
)

print(f"Dimensions del dataset llarg: {df_llarg.shape}")
df_llarg.head(10)

Dimensions del dataset llarg: (7896, 13)


,id_aleatori,Nom,Cognoms,DNI,País d'origen,Ciutat,Gènere,Grup Professional,Càrrec,franja_edat,categoria_cost_vida,variable,valor
0,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,Grup B,Cap de projecte,60+,low,Salari mensual,1469.0
1,1,Marc,Muñoz,48840994W,Espanya,Alacant,Home,Grup C,Senior analyst,60+,medium,Salari mensual,2718.0
2,2,Noa,Serra,14308421X,Espanya,Alacant,Dona,Grup A,Tècnic IT,60+,medium,Salari mensual,1358.0
3,3,Pol,Gil,58586340F,Espanya,Sevilla,Home,Grup B,Data Analyst,30-44,medium,Salari mensual,1478.0
4,4,David,Vila,82070937P,Espanya,Bilbao,Dona,Grup B,Administratiu,60+,medium,Salari mensual,1284.0
5,5,Alexia,Puig,25014631B,Espanya,Las Palmas de Gran Canaria,Dona,Grup B,Cap de projecte,45-59,low,Salari mensual,2069.0
6,6,David,Gómez Vila,36998038E,Espanya,Palma,Dona,Grup B,Cap de projecte,45-59,medium,Salari mensual,2349.0
7,7,Carmen,López,47376585K,Espanya,Barcelona,Home,Grup C,Responsable de vendes,60+,medium,Salari mensual,2855.0
8,540,Nora,Andersen Pedersen,37289414J,Noruega,Stavanger,Dona,Grup B,Analista junior,60+,NaN,Salari mensual,1338.0
9,9,Alba,García,75181648S,Espanya,Las Palmas de Gran Canaria,Home,Grup A,Senior analyst,60+,low,Salari mensual,1450.0


**Per què el format llarg facilita les comparacions**

- En el format ample, cada mesura (salari, km anuals, consum, temperatura...) és una columna diferent amb la seva pròpia escala i unitat, cosa que dificulta calcular estadístics agregats sobre "totes les mesures" de cop o comparar-les entre si.
- En el format llarg, totes les mesures conviuen en dues úniques columnes (`variable` i `valor`), de manera que es pot filtrar, agrupar o agregar per tipus de variable amb un únic `groupby('variable')`, sense haver d'escriure una línia de codi diferent per a cada columna original.
- Aquest format és el que esperen la majoria d'eines de visualització (com Seaborn) per generar gràfics comparatius entre variables de manera automàtica, cosa que aprofitarem al Nivell 3.
- El dataset llarg té 7896 files (987 persones × 8 variables mesurades), confirmant que la transformació no ha perdut ni duplicat cap dada.

### Exercici 3: Creació de variables derivades (Feature Engineering)

A partir de la informació ja disponible, es poden construir indicadors que aportin més valor analític que les variables originals per si soles, útils per identificar patrons de risc o prioritats de gestió de personal.

**Algoritme:**
1. Creo un índex de dependència familiar (salari respecte al nombre de persones a càrrec).
2. Creo un índex de mobilitat (quilometratge anual respecte al clima de la ciutat).
3. Creo una ràtio de salari ajustat respecte al cost de vida (poder adquisitiu real).
4. Reviso cada variable nova per detectar casos límit o valors incoherents generats per la pròpia transformació.

In [52]:
# índex de dependència familiar: salari disponible per persona a càrrec (el +1 evita dividir per zero i inclou la pròpia persona)
df_empleats['index_dependencia_familiar'] = df_empleats['Salari mensual'] / (df_empleats['Nombre_fills'].fillna(0) + 1)

# índex de mobilitat: quilòmetres anuals en relació amb la temperatura mitjana de la ciutat
df_empleats['index_mobilitat'] = df_empleats['Km_anuals'] / df_empleats['Temperatura_mitjana_ciutat']

# ràtio de poder adquisitiu: salari ajustat en relació amb el cost de vida de la ciutat
df_empleats['ratio_salari_cost_vida'] = df_empleats['salari_ajustat'] / df_empleats['Cost_vida_index']

df_empleats[['index_dependencia_familiar', 'index_mobilitat', 'ratio_salari_cost_vida']].describe()

,index_dependencia_familiar,index_mobilitat,ratio_salari_cost_vida
count,987.000000,311.000000,855.000000
mean,1178.288061,3655.848041,23.760343
std,690.716780,4907.256129,9.803553
min,143.600000,-16000.000000,9.375875
25%,641.625000,1337.425940,16.453013
50%,1086.000000,3225.806452,21.302885
75%,1562.000000,5304.427675,28.274005
max,3672.000000,58479.532164,65.148387


In [53]:
# localitzo els casos amb índex de mobilitat negatiu, causats per una temperatura registrada negativa
casos_mobilitat_negativa = df_empleats[df_empleats['index_mobilitat'] < 0][['Ciutat', 'Km_anuals', 'Temperatura_mitjana_ciutat', 'index_mobilitat']]
print(f"Casos amb índex de mobilitat negatiu: {len(casos_mobilitat_negativa)}")
casos_mobilitat_negativa

Casos amb índex de mobilitat negatiu: 13


,Ciutat,Km_anuals,Temperatura_mitjana_ciutat,index_mobilitat
349,Köln,43566.0,-5.0,-8713.200000
623,Firenze,80000.0,-5.0,-16000.000000
655,Bordeaux,65983.0,-5.0,-13196.600000
664,Tarragona,50000.0,-5.0,-10000.000000
727,Alacant,80000.0,-10.0,-8000.000000
728,Sevilla,-50.0,20.0,-2.500000
750,Berlin,-50.0,10.7,-4.672897
765,A Coruña,-50.0,13.8,-3.623188
801,Madrid,-50.0,42.0,-1.190476
830,Valladolid,-50.0,12.7,-3.937008


In [54]:
# un quilometratge anual negatiu no té sentit físic: el tracto com a valor desconegut, no com un error de signe a corregir
df_empleats.loc[df_empleats['Km_anuals'] < 0, 'Km_anuals'] = np.nan

In [55]:
# recalculo l'índex de mobilitat ara que Km_anuals negatiu ja és NaN
df_empleats['index_mobilitat'] = df_empleats['Km_anuals'] / df_empleats['Temperatura_mitjana_ciutat']

# per als 8 casos restants de temperatura negativa, l'índex encara no té una interpretació física vàlida (km/grau amb grau negatiu),
# així que el marco explícitament com a no interpretable en lloc de deixar un número enganyós
df_empleats.loc[df_empleats['Temperatura_mitjana_ciutat'] < 0, 'index_mobilitat'] = np.nan

# confirmo que ja no queda cap valor negatiu
print(f"Valors negatius restants: {(df_empleats['index_mobilitat'] < 0).sum()}")

Valors negatius restants: 0


In [56]:
# reviso si índex_dependencia_familiar o ratio_salari_cost_vida tenen valors extrems o no interpretables
print("Índex de dependència familiar:")
print(df_empleats['index_dependencia_familiar'].describe())

print("\nRàtio salari / cost de vida:")
print(df_empleats['ratio_salari_cost_vida'].describe())

Índex de dependència familiar:
count     987.000000
mean     1178.288061
std       690.716780
min       143.600000
25%       641.625000
50%      1086.000000
75%      1562.000000
max      3672.000000
Name: index_dependencia_familiar, dtype: float64

Ràtio salari / cost de vida:
count    855.000000
mean      23.760343
std        9.803553
min        9.375875
25%       16.453013
50%       21.302885
75%       28.274005
max       65.148387
Name: ratio_salari_cost_vida, dtype: float64


**Variables derivades creades i decisions preses**

- **Índex de dependència familiar** (`Salari mensual / (Nombre_fills + 1)`): indica quants euros de salari "corresponen" a cada membre a càrrec, útil per detectar situacions de major pressió econòmica familiar. Els casos amb `Nombre_fills` desconegut s'han tractat com a 0 fills únicament a efectes d'aquest càlcul, una assumpció que infla lleugerament l'índex en aquests casos i que cal tenir present en la seva interpretació.
- **Índex de mobilitat** (`Km_anuals / Temperatura_mitjana_ciutat`): relaciona el desplaçament anual amb el clima de la ciutat. S'han detectat 13 casos límit: 5 amb `Km_anuals` negatiu (valor sense sentit físic, convertit a mancant en lloc d'assumir-ne el signe) i 8 amb temperatura negativa registrada, on l'índex resultant no té una interpretació física vàlida i s'ha marcat explícitament com a no interpretable, sense alterar la temperatura original ja validada al Nivell 1.
- **Ràtio salari / cost de vida** (`salari_ajustat / Cost_vida_index`): aproxima el poder adquisitiu real de cada persona segons on viu. No es calcula per a les 132 persones sense informació de cost de vida de ciutat, coherent amb el criteri seguit des de l'Exercici 1 del Nivell 2.

### Exercici 4: Exportació i síntesi de resultats

Un cop preparades i enriquides les dades, cal deixar-les organitzades en sortides reutilitzables: un fitxer detallat per a cada Grup Professional i un resum agregat que permeti una lectura ràpida sense haver de consultar el detall complet.

**Algoritme:**
1. Recorro cada Grup Professional amb un bucle i exporto les seves dades a un fitxer independent.
2. Construeixo un dataframe resum amb el nombre de treballadors, el salari mitjà i l'edat mediana per Grup Professional, i l'exporto.

In [57]:
# recorro cada grup professional únic i exporto les seves dades a un fitxer separat
for grup in df_empleats['Grup Professional'].dropna().unique():
    df_grup = df_empleats[df_empleats['Grup Professional'] == grup]
    nom_fitxer = f"dades_{grup.replace(' ', '_')}.csv"
    df_grup.to_csv(nom_fitxer, index=False)
    print(f"Exportat {nom_fitxer}: {len(df_grup)} files")

Exportat dades_Grup_B.csv: 351 files
Exportat dades_Grup_C.csv: 140 files
Exportat dades_Grup_A.csv: 456 files
Exportat dades_Grup_D.csv: 40 files


In [58]:
# construeixo el resum agregat: nombre de treballadors, salari mitjà i edat mediana per grup
resum_grups = df_empleats.groupby('Grup Professional').agg(
    nombre_treballadors=('id_aleatori', 'count'),
    salari_mitja=('Salari mensual', 'mean'),
    edat_mediana=('edat', 'median')
).reset_index()

resum_grups.to_csv('resum_grups_professionals.csv', index=False)
resum_grups

,Grup Professional,nombre_treballadors,salari_mitja,edat_mediana
0,Grup A,456,1101.870614,44.0
1,Grup B,351,1639.767806,44.0
2,Grup C,140,2377.735714,46.0
3,Grup D,40,3166.625000,38.0


In [59]:
# exporto tots els grups i el resum a un únic fitxer Excel, cada un en la seva pròpia pestanya
with pd.ExcelWriter('dades_per_grup_professional.xlsx') as writer:
    for grup in df_empleats['Grup Professional'].dropna().unique():
        df_grup = df_empleats[df_empleats['Grup Professional'] == grup]
        nom_pestanya = grup.replace(' ', '_')
        df_grup.to_excel(writer, sheet_name=nom_pestanya, index=False)

    resum_grups.to_excel(writer, sheet_name='Resum', index=False)

print("Fitxer Excel generat amb totes les pestanyes")

Fitxer Excel generat amb totes les pestanyes


**Resum de l'exportació**

- S'han exportat 4 fitxers CSV independents, un per Grup Professional (A: 456 files, B: 351, C: 140, D: 40), que sumen les 987 files totals del dataset, confirmant que cap registre s'ha perdut ni duplicat en el procés.
- S'ha generat un cinquè fitxer amb el resum agregat per grup (nombre de treballadors, salari mitjà i edat mediana), que permet una lectura ràpida sense consultar el detall complet: el Grup D és el més petit però amb el salari mitjà més alt i la població més jove; el Grup A és el més nombrós amb el salari mitjà més baix.
- (Extra) S'ha generat també un únic fitxer Excel amb una pestanya per cada Grup Professional més una pestanya addicional amb el resum agregat, consolidant totes les sortides en un sol document.

## Nivell 3 - Exploració visual i optimització aplicada

### Exercici 1: Exploració visual del dataset

Amb un volum elevat de variables, cal disposar d'una manera ràpida i sistemàtica de visualitzar-les totes sense haver de repetir manualment el mateix procés columna per columna.

**Algoritme:**
1. Defineixo una funció que, per a cada columna d'un dataframe, detecta si és numèrica, categòrica o de tipus data.
2. Segons el tipus, genera el gràfic més adequat: histograma/boxplot per numèriques, barres de freqüència per categòriques, barres per anys si és una data.
3. Exporto cada gràfic a un fitxer.
4. Provo la funció amb un dataset d'exemple de seaborn (per exemple, `penguins`).

In [60]:
def explora_visualment(df, carpeta_sortida='graphics'):
    # creo la carpeta de sortida si encara no existeix
    os.makedirs(carpeta_sortida, exist_ok=True)

    for columna in df.columns:
        serie = df[columna]

        # cas 1: columna de tipus data
        if pd.api.types.is_datetime64_any_dtype(serie):
            fig, ax = plt.subplots()
            serie.dt.year.value_counts().sort_index().plot(kind='bar', ax=ax)
            ax.set_title(f'Freqüència per any — {columna}')

        # cas 2: columna numèrica
        elif pd.api.types.is_numeric_dtype(serie):
            fig, axes = plt.subplots(1, 2, figsize=(10, 4))
            serie.plot(kind='hist', ax=axes[0], title=f'Histograma — {columna}')
            serie.plot(kind='box', ax=axes[1], title=f'Boxplot — {columna}')

        # cas 3: la resto de columnes es tracten com a categòriques
        else:
            fig, ax = plt.subplots()
            serie.value_counts().head(15).plot(kind='bar', ax=ax)
            ax.set_title(f'Valors més freqüents — {columna}')

        # deso el gràfic a un fitxer amb el nom de la columna i tanco la figura per no acumular memòria
        fig.tight_layout()
        fig.savefig(os.path.join(carpeta_sortida, f'{columna}.png'))
        plt.close(fig)

    print(f"Gràfics generats a la carpeta '{carpeta_sortida}'")

In [61]:
df_penguins = sns.load_dataset('penguins')
explora_visualment(df_penguins)

Gràfics generats a la carpeta 'graphics'


**Funció d'exploració visual automatitzada**

- S'ha creat una funció genèrica que recorre totes les columnes d'un dataframe i genera automàticament el gràfic més adequat segons el tipus de dada: histograma i boxplot per a columnes numèriques, gràfic de barres dels valors més freqüents per a categòriques, i barres per any per a columnes de tipus data.
- Cada gràfic es desa com a fitxer `.png` independent dins d'una carpeta de sortida, amb el nom de la columna corresponent.
- S'ha provat amb el dataset `penguins` de seaborn, generant correctament els 7 gràfics corresponents a les seves 7 columnes, cadascun amb el format visual adequat al seu tipus de dada.

### Exercici 2. Optimització de rutes (problema aplicat)

L'empresa vol fer una prova pilot de control de desplaçaments amb els treballadors ubicats a Espanya, a partir d'una matriu de distàncies entre ciutats. No es busca la ruta òptima, sinó una aproximació senzilla i explicable.

**Algoritme:**
1. Carrego la matriu de distàncies, amb les ciutats com a files i columnes, i elimino "Las Palmas de Gran Canaria" i "Palma".
2. Defineixo una funció que, donades una ciutat d'origen i una de destinació, construeixi una ruta seguint sempre la ciutat no visitada més propera, fins a arribar a la destinació.
3. La funció retorna l'ordre de ciutats visitades i la distància total recorreguda.

In [62]:
# carrego la matriu de distàncies, amb les ciutats com a índex (files) i com a columnes
df_distancies = pd.read_excel(r'C:\Users\FDR\Documents\IT_Academy\Especialitat_en_Data_Analytics\sprint8\data\matriu_distancies.xlsx', index_col=0)

# elimino "Las Palmas de Gran Canaria" i "Palma" tant de files com de columnes
ciutats_excloure = ['Las Palmas de Gran Canaria', 'Palma']
df_distancies = df_distancies.drop(index=ciutats_excloure, columns=ciutats_excloure, errors='ignore')

df_distancies.head()

,Barcelona,Valencia,Sevilla,Zaragoza,Málaga,Murcia,Bilbao,Alicante,Córdoba,Valladolid,Vigo,Gijón,Hospitalet de Llobregat
Barcelona,NaN,303.0,831.0,256.0,770.0,471.0,469.0,407.0,711.0,576.0,908.0,686.0,7.0
Valencia,303.0,NaN,541.0,246.0,468.0,177.0,473.0,125.0,421.0,441.0,766.0,632.0,297.0
Sevilla,831.0,541.0,NaN,646.0,158.0,433.0,703.0,495.0,121.0,486.0,588.0,685.0,824.0
Zaragoza,256.0,246.0,646.0,NaN,628.0,408.0,246.0,368.0,535.0,320.0,652.0,444.0,250.0
Málaga,770.0,468.0,158.0,628.0,NaN,323.0,739.0,391.0,133.0,549.0,716.0,766.0,763.0


In [63]:
def construeix_ruta(origen, desti, matriu):
    ciutat_actual = origen
    ruta = [ciutat_actual]
    distancia_total = 0
    ciutats_per_visitar = list(matriu.columns)
    ciutats_per_visitar.remove(ciutat_actual)

    # continuo afegint ciutats fins que la ciutat actual sigui la destinació
    while ciutat_actual != desti:
        # distàncies de la ciutat actual a totes les ciutats encara no visitades
        distancies_disponibles = matriu.loc[ciutat_actual, ciutats_per_visitar]

        # trio la ciutat no visitada més propera
        seguent_ciutat = distancies_disponibles.idxmin()
        distancia_seguent = distancies_disponibles.min()

        ruta.append(seguent_ciutat)
        distancia_total += distancia_seguent
        ciutats_per_visitar.remove(seguent_ciutat)
        ciutat_actual = seguent_ciutat

    return ruta, distancia_total

In [64]:
ruta, distancia = construeix_ruta('Barcelona', 'Sevilla', df_distancies)
print(f"Ruta: {' → '.join(ruta)}")
print(f"Distància total: {distancia} km")

Ruta: Barcelona → Hospitalet de Llobregat → Zaragoza → Valencia → Alicante → Murcia → Córdoba → Sevilla
Distància total: 1138.0 km


**Estratègia i limitacions de la ruta**

- S'ha implementat l'estratègia del "veí més proper": a cada pas, des de la ciutat actual es tria sempre la ciutat no visitada més propera, fins arribar a la destinació.
- Amb origen Barcelona i destinació Sevilla, la ruta resultant és Barcelona → Hospitalet de Llobregat → Zaragoza → Valencia → Alicante → Murcia → Córdoba → Sevilla, amb una distància total de 1138 km.
- Aquesta distància és superior al trajecte directe (831 km, un 37% més llarg), ja que l'estratègia prioritza sempre el pas més curt immediat sense tenir en compte el recorregut global — és una limitació coneguda d'aquest tipus d'heurística ("veí més proper"), acceptable per a una prova pilot senzilla i explicable, tal com demana l'enunciat, però no equivalent a una ruta òptima.
- "Las Palmas de Gran Canaria" i "Palma" s'han exclòs de la matriu tal com indica l'enunciat, en no formar part de la prova pilot amb treballadors a Espanya peninsular.